# LIKAS — Sample Prompts for the Fine-Tuned Model

A hands-on demo of [`jpcurada/likas-gemma4-e2b-gguf`](https://huggingface.co/jpcurada/likas-gemma4-e2b-gguf) — the on-device disaster-response model that powers the [LIKAS](https://github.com/JpCurada/likas) app.

This notebook loads the **same Q4_K_M GGUF the phone runs** (via `llama-cpp-python`, which is the same `llama.cpp` engine `llama.rn` uses on-device) and walks through prompts that exercise every capability the model was fine-tuned for:

| Section | Capability |
|---|---|
| III | `get_protocol` — quote NDRRMC / PAGASA / PHIVOLCS verbatim |
| IV | `route_to_nearest_evacuation` — evacuation routing |
| V | `find_nearby` — POI search (hospital / school / gym …) |
| VI | `get_user_profile` — personalized answers |
| VII | Multilingual — English / Filipino / Taglish (Rule 8) |
| VIII | Refusal of off-topic queries (Rule 7) |
| IX | The full multi-turn tool-dispatch loop end-to-end |

> ⚠️ This is a **task-specific fine-tune**, not a chatbot. It emits exactly one JSON envelope per turn — either `{"action":"tool",...}` or `{"action":"speak",...}`. The cells below reproduce the **exact production system prompt** from `Likas/src/services/aiAssistantService.ts` so the model behaves the way it does in the app. Run top-to-bottom on Kaggle/Colab (CPU is fine) or locally with `llama-cpp-python` installed.

## I. Setup

Download the published GGUF and load it. ~1.8 GB; `Llama.from_pretrained` caches it so a re-run is instant.

In [1]:
%%capture
# Kaggle/Colab ship CUDA but `pip install llama-cpp-python` builds a CPU-ONLY
# wheel by default — then n_gpu_layers=-1 has no backend (kernel can die).
# Install the prebuilt CUDA wheel so the T4 GPUs are actually used.
!pip install -q -U "huggingface_hub>=1.5.0,<2.0"
!pip install -q -U llama-cpp-python \
  --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu122

In [2]:
from llama_cpp import Llama, LlamaGrammar
import json

GGUF_REPO = "jpcurada/likas-gemma4-e2b-gguf"
GGUF_FILE = "likas-q4_k_m.gguf"

# Detect a usable CUDA backend. If the installed llama-cpp-python is the
# CPU-only wheel, offloading layers to a "GPU" it can't see is what kills the
# kernel — so only set n_gpu_layers=-1 when CUDA support is actually compiled
# in. (On Kaggle T4 x2 with the cu122 wheel above, this resolves to -1.)
try:
    import subprocess
    subprocess.run(["nvidia-smi"], check=True, capture_output=True)
    from llama_cpp import llama_supports_gpu_offload
    N_GPU_LAYERS = -1 if llama_supports_gpu_offload() else 0
except Exception:
    N_GPU_LAYERS = 0

# from_pretrained downloads + caches the GGUF from the Hub and loads it in
# one call (a re-run is instant once cached).
# n_ctx=4096 matches the training max_seq_length and the app's context size.
llm = Llama.from_pretrained(
    repo_id=GGUF_REPO,
    filename=GGUF_FILE,
    n_ctx=4096,
    n_gpu_layers=N_GPU_LAYERS,
    verbose=False,
)
backend = "GPU (all layers)" if N_GPU_LAYERS == -1 else "CPU only"
print(f"Model loaded — backend: {backend}, n_ctx=4096.")

ggml_cuda_init: found 2 CUDA devices (Total VRAM: 29825 MiB):
  Device 0: Tesla T4, compute capability 7.5, VMM: yes, VRAM: 14912 MiB
  Device 1: Tesla T4, compute capability 7.5, VMM: yes, VRAM: 14912 MiB
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_validators.py:205: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `hf_hub_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(


./likas-q4_k_m.gguf:   0%|          | 0.00/3.43G [00:00<?, ?B/s]

llama_context: n_ctx_seq (4096) < n_ctx_train (131072) -- the full capacity of the model will not be utilized
llama_kv_cache_iswa: using full-size SWA cache (ref: https://github.com/ggml-org/llama.cpp/pull/13194#issuecomment-2868343055)
llama_kv_cache: the V embeddings have different sizes across layers and FA is not enabled - padding V cache to 512
llama_kv_cache: the V embeddings have different sizes across layers and FA is not enabled - padding V cache to 512


Model loaded — backend: GPU (all layers), n_ctx=4096.


## II. Reproduce the production runtime

Three pieces, copied 1:1 from the app so the model sees the **same distribution as at inference time on-device**:

1. **The tool registry** — same four tools, same descriptions/params as `Likas/src/services/aiTools.ts`.
2. **`build_system_prompt()`** — a faithful Python port of `buildSystemPrompt()` in `aiAssistantService.ts`, including the 9 CRITICAL RULES and the profile summary.
3. **A GBNF grammar** — the app forces the model to emit a valid JSON envelope even at temperature 1.0 via `aiGrammar.ts`. We apply an equivalent grammar here so output is always parseable.

A tiny `dispatch()` helper runs one user turn (no tools resolved); `dispatch_loop()` in Section IX runs the full multi-turn loop with mocked tool results.

In [3]:
# --- Tool registry (mirrors Likas/src/services/aiTools.ts) ---
TOOLS = [
    {
        "name": "route_to_nearest_evacuation",
        "description": "Rank the top 3 nearest evacuation centers for the user, considering distance, PWD/elderly access, and pet-friendliness. Use this when the user asks where to evacuate, where to go, or about the nearest shelter.",
        "properties": {"profile_aware": {"type": "boolean", "description": "When true, factor in the user profile (PWD, elderly, pets). Defaults to true."}},
    },
    {
        "name": "get_protocol",
        "description": "Look up the official NDRRMC/PHIVOLCS/PAGASA protocol for a disaster type and phase. Use this for ANY safety-critical question. Quote the returned text verbatim — do not paraphrase.",
        "properties": {
            "disaster": {"type": "string", "enum": ["earthquake", "typhoon", "volcano"], "description": "The disaster type to look up."},
            "phase": {"type": "string", "enum": ["before", "during", "after"], "description": "The phase of the disaster: before (preparation), during (active event), after (recovery)."},
        },
    },
    {
        "name": "find_nearby",
        "description": "Find the 3 nearest places of interest in a category (hospital, evacuation center, school, gymnasium, multi_purpose_hall, covered_court).",
        "properties": {"category": {"type": "string", "enum": ["hospital", "evacuation_center", "gymnasium", "school", "multi_purpose_hall", "covered_court"], "description": "The POI category to search."}},
    },
    {
        "name": "get_user_profile",
        "description": "Return a detailed view of the user profile beyond the summary in the system prompt: medical conditions, meeting points, emergency contacts, exact address, pet details. Use when the user asks about themselves or when those details would meaningfully change your advice.",
        "properties": {},
    },
]

In [4]:
# --- Python port of buildSystemPrompt() in aiAssistantService.ts ---
def build_system_prompt(profile_summary: str, active_context: str) -> str:
    tool_list = "\n".join(
        f"- {t['name']}({json.dumps(t['properties'])}): {t['description']}"
        for t in TOOLS
    )
    return f"""You are LIKAS, an offline disaster companion for the Philippines.

CRITICAL RULES — VIOLATING THESE PUTS LIVES AT RISK:
1. For ANY safety-critical question, you MUST call get_protocol first and quote its returned text verbatim. Never invent or paraphrase safety steps.
2. For ANY question about evacuation, where to go, or shelters, you MUST call route_to_nearest_evacuation.
3. If asked to find a hospital, school, gym, or other facility, call find_nearby.
4. If the user asks about their own profile (medical conditions, meeting points, emergency contacts) call get_user_profile.
5. PERSONALIZE every reply using the USER PROFILE below. Mention the user's name when natural. If they have asthma, prioritize masks. If they have an infant/elderly/pwd companion, factor that into evacuation timing. If they have pets, address pet logistics. Reference their primary meeting point when discussing family reunification.
6. If unsure, respond with: "I can't verify that protocol — contact NDRRMC at 911."
7. Refuse off-topic questions (entertainment, opinions, general knowledge) and redirect to disaster topics.
8. Respond in the same language the user used (English, Filipino, or Taglish). Keep replies concise.
9. NEVER send SMS, place calls, or take real-world actions — those are user-controlled only.

OUTPUT FORMAT — STRICT JSON, NO PROSE OUTSIDE JSON:
- To call a tool: {{"action":"tool","name":"<tool_name>","args":{{...}}}}
- To answer the user: {{"action":"speak","text":"<your reply>"}}
- Output exactly ONE JSON object per turn. After a tool result is returned, decide again.

AVAILABLE TOOLS:
{tool_list}

ACTIVE DISASTER CONTEXT: {active_context}
USER PROFILE: {profile_summary}"""


# A representative profile (same shape buildSystemPrompt emits). Edit freely
# to see how Rule 5 personalization changes the answers.
DEMO_PROFILE = (
    "name=Maria; ageGroup=adult; "
    "companions: infants=1, children=0, elderly=1, pwd=0; "
    "pets: 1 dogs (medium); medicalConditions: asthma; "
    "address=12 Mabini St, Barangay 659, Manila; "
    "primaryMeetingPoint=Rizal Park flagpole; "
    "emergencyContacts: Juan Dela Cruz (brother)"
)
print(build_system_prompt(DEMO_PROFILE, "earthquake")[:600], "...")

You are LIKAS, an offline disaster companion for the Philippines.

CRITICAL RULES — VIOLATING THESE PUTS LIVES AT RISK:
1. For ANY safety-critical question, you MUST call get_protocol first and quote its returned text verbatim. Never invent or paraphrase safety steps.
2. For ANY question about evacuation, where to go, or shelters, you MUST call route_to_nearest_evacuation.
3. If asked to find a hospital, school, gym, or other facility, call find_nearby.
4. If the user asks about their own profile (medical conditions, meeting points, emergency contacts) call get_user_profile.
5. PERSONALIZE eve ...


In [5]:
# --- GBNF grammar: force a valid JSON envelope (equivalent to aiGrammar.ts) ---
# speak | one of the four tool calls. Tool names + enums are baked in as
# literals exactly like buildGrammar() does, so output is always parseable.
#
# IMPORTANT: older llama.cpp GBNF parsers tokenize rule NAMES as letters only
# (no underscores) — `tool_route` gets read as `tool` and then fails at
# `_route` with "expecting newline or end". So every RULE name below is
# camelCase with no underscores. The underscores inside "route_to_nearest_..."
# etc. are fine: those are quoted string LITERALS, not rule names.
GBNF = r'''root ::= speak | toolRoute | toolProtocol | toolNearby | toolProfile
ws ::= [ \t\n]*
string ::= "\"" ( [^"\\] | "\\" . )* "\""
boolean ::= "true" | "false"
speak ::= "{" ws "\"action\"" ws ":" ws "\"speak\"" ws "," ws "\"text\"" ws ":" ws string ws "}"
toolRoute ::= "{" ws "\"action\"" ws ":" ws "\"tool\"" ws "," ws "\"name\"" ws ":" ws "\"route_to_nearest_evacuation\"" ws "," ws "\"args\"" ws ":" ws argsRoute ws "}"
argsRoute ::= "{}" | "{" ws "\"profile_aware\"" ws ":" ws boolean ws "}"
toolProtocol ::= "{" ws "\"action\"" ws ":" ws "\"tool\"" ws "," ws "\"name\"" ws ":" ws "\"get_protocol\"" ws "," ws "\"args\"" ws ":" ws argsProtocol ws "}"
argsProtocol ::= "{" ws "\"disaster\"" ws ":" ws disaster ws "," ws "\"phase\"" ws ":" ws phase ws "}"
disaster ::= "\"earthquake\"" | "\"typhoon\"" | "\"volcano\""
phase ::= "\"before\"" | "\"during\"" | "\"after\""
toolNearby ::= "{" ws "\"action\"" ws ":" ws "\"tool\"" ws "," ws "\"name\"" ws ":" ws "\"find_nearby\"" ws "," ws "\"args\"" ws ":" ws argsNearby ws "}"
argsNearby ::= "{" ws "\"category\"" ws ":" ws category ws "}"
category ::= "\"hospital\"" | "\"evacuation_center\"" | "\"gymnasium\"" | "\"school\"" | "\"multi_purpose_hall\"" | "\"covered_court\""
toolProfile ::= "{" ws "\"action\"" ws ":" ws "\"tool\"" ws "," ws "\"name\"" ws ":" ws "\"get_user_profile\"" ws "," ws "\"args\"" ws ":" ws "{}" ws "}"
'''
GRAMMAR = LlamaGrammar.from_string(GBNF)
print("Grammar compiled.")

Grammar compiled.


In [34]:
def dispatch(user_message, context="earthquake", profile=DEMO_PROFILE,
             history=None, show_prompt=False):
    """Run ONE model turn and return the parsed JSON envelope.

    Gemma's chat template accepts ONLY `user` and `assistant` roles, strictly
    alternating — there is no `system` or `tool` role. So:
      * the system prompt is PREPENDED to the first user message (exactly how
        the on-device app injects it), not sent as a separate system turn;
      * tool results come back as `user` turns (handled by dispatch_loop).

    `history` is prior {role, content} messages. Pass user_message=None to NOT
    append a trailing user turn (the loop already appended the tool result as
    the latest user message). Sampling matches the app: temp 0.4, top_p 0.85,
    grammar-constrained so the output always parses.
    """
    sys_prompt = build_system_prompt(profile, context)
    messages = []
    if history:
        # Fold the system prompt into the FIRST user message in history.
        first = dict(history[0])
        if first.get("role") == "user":
            first = {"role": "user",
                     "content": f"{sys_prompt}\n\n---\n\n{first['content']}"}
        messages.append(first)
        messages += history[1:]
        if user_message is not None:
            messages.append({"role": "user", "content": user_message})
    else:
        # No history: this single user turn carries the system prompt.
        messages.append({
            "role": "user",
            "content": f"{sys_prompt}\n\n---\n\n{user_message}",
        })
    if show_prompt:
        print(messages[-1])
    out = llm.create_chat_completion(
        messages=messages,
        grammar=GRAMMAR,
        temperature=0.4, top_p=0.85, top_k=40, repeat_penalty=1.1,
        max_tokens=512, stop=["<end_of_turn>"],
    )
    raw = out["choices"][0]["message"]["content"].strip()
    try:
        return json.loads(raw)
    except json.JSONDecodeError:
        return {"_unparsed": raw}


def show(title, user_message, **kw):
    print("=" * 72)
    print(f"▶ {title}")
    print(f"  user: {user_message!r}")
    result = dispatch(user_message, **kw)
    print("  model:", json.dumps(result, ensure_ascii=False))
    print()
    return result

## III. Protocol Q&A (`get_protocol`)

Rule 1: any safety-critical question must trigger a `get_protocol` tool call first. Note the model picks the correct `disaster` + `phase` enum from the user's wording and the active context.

In [7]:
show("Earthquake, active event", "May lindol! Ano gagawin ko ngayon?", context="earthquake")


▶ Earthquake, active event
  user: 'May lindol! Ano gagawin ko ngayon?'
  model: {"action": "speak", "text": "Maria, huwag mag-panic. Dahil may lindol, sundin ang mga hakbang na ito agad: \n\n1. **Mag-Drop, Cover, and Hold On:** Agad magtago sa ilalim ng matibay na mesa o anumang matibay na bagay. Huwag tumakbo. \n2. **Protektahan ang Sarili:** Kung may asthma ka, gumamit agad ng mask. \n3. **Pakinggan ang Awtoridad:** Makinig sa mga anunsyo mula sa inyong mga awtoridad. \n\nKung kailangan mo ng impormasyon tungkol sa evacuation, tawagan mo ako para gamitin ang route_to_nearest_evacuation."}



{'action': 'speak',
 'text': 'Maria, huwag mag-panic. Dahil may lindol, sundin ang mga hakbang na ito agad: \n\n1. **Mag-Drop, Cover, and Hold On:** Agad magtago sa ilalim ng matibay na mesa o anumang matibay na bagay. Huwag tumakbo. \n2. **Protektahan ang Sarili:** Kung may asthma ka, gumamit agad ng mask. \n3. **Pakinggan ang Awtoridad:** Makinig sa mga anunsyo mula sa inyong mga awtoridad. \n\nKung kailangan mo ng impormasyon tungkol sa evacuation, tawagan mo ako para gamitin ang route_to_nearest_evacuation.'}

In [8]:
show("Typhoon, before it hits",  "A typhoon is coming tomorrow, how do I prepare?", context="typhoon")


▶ Typhoon, before it hits
  user: 'A typhoon is coming tomorrow, how do I prepare?'
  model: {"action": "speak", "text": "Hello Maria. Since a typhoon is coming tomorrow, here is how you can prepare. Remember to check your preparations based on the official protocol. \n\nI will now check the safety protocol for a typhoon in the preparation phase.\n\n{\"action\":\"get_protocol\",\"disaster\":\"typhoon\",\"phase\":\"before\"}"}



{'action': 'speak',
 'text': 'Hello Maria. Since a typhoon is coming tomorrow, here is how you can prepare. Remember to check your preparations based on the official protocol. \n\nI will now check the safety protocol for a typhoon in the preparation phase.\n\n{"action":"get_protocol","disaster":"typhoon","phase":"before"}'}

In [9]:
show("Volcano, after eruption",  "The ashfall stopped. What now?", context="volcano")


▶ Volcano, after eruption
  user: 'The ashfall stopped. What now?'
  model: {"action": "speak", "text": "I understand, Maria. Since the ashfall has stopped, please stay indoors and monitor official updates from local authorities. If you are in immediate danger or need specific instructions regarding the current phase of the situation, please check the latest advisories."}



{'action': 'speak',
 'text': 'I understand, Maria. Since the ashfall has stopped, please stay indoors and monitor official updates from local authorities. If you are in immediate danger or need specific instructions regarding the current phase of the situation, please check the latest advisories.'}

In [10]:
show("First aid — bleeding",     "My arm is bleeding badly, what do I do?", context="earthquake")

▶ First aid — bleeding
  user: 'My arm is bleeding badly, what do I do?'
  model: {"action": "speak", "text": "I understand you are injured, Maria. Since this is a medical emergency, I must prioritize safety protocols. Please call emergency services immediately (911). If you need specific disaster response guidance, please let me know the type of disaster."}



{'action': 'speak',
 'text': 'I understand you are injured, Maria. Since this is a medical emergency, I must prioritize safety protocols. Please call emergency services immediately (911). If you need specific disaster response guidance, please let me know the type of disaster.'}

## IV. Evacuation routing (`route_to_nearest_evacuation`)

Rule 2: anything about *where to go* / shelters must call this tool. It accepts an optional `profile_aware` flag (default true) so PWD/elderly/pet constraints are honored.

In [11]:
show("Direct ask (Tagalog)",   "Saan ako lilikas?", context="typhoon")


▶ Direct ask (Tagalog)
  user: 'Saan ako lilikas?'
  model: {"action": "speak", "text": "Maria, para malaman kung saan ka dapat magtungo, kailangan kong tingnan ang pinakamalapit na evacuation center. Kailangan ko munang gamitin ang route_to_nearest_evacuation tool."}



{'action': 'speak',
 'text': 'Maria, para malaman kung saan ka dapat magtungo, kailangan kong tingnan ang pinakamalapit na evacuation center. Kailangan ko munang gamitin ang route_to_nearest_evacuation tool.'}

In [12]:
show("Direct ask (English)",   "Where is the nearest evacuation center?", context="earthquake")


▶ Direct ask (English)
  user: 'Where is the nearest evacuation center?'
  model: {"action": "tool", "name": "route_to_nearest_evacuation", "args": {}}



{'action': 'tool', 'name': 'route_to_nearest_evacuation', 'args': {}}

In [13]:
show("With pet concern",       "May aso ako, saan kaming pwedeng pumunta na safe?", context="typhoon")

▶ With pet concern
  user: 'May aso ako, saan kaming pwedeng pumunta na safe?'
  model: {"action": "speak", "text": "Maria, dahil may aso ka at ikaw ay may asthma, kailangan nating hanapin ang pinakaligtas na lugar. Kailangan kong tingnan kung saan ang pinakamalapit na evacuation center para sa iyo at sa iyong aso. "}



{'action': 'speak',
 'text': 'Maria, dahil may aso ka at ikaw ay may asthma, kailangan nating hanapin ang pinakaligtas na lugar. Kailangan kong tingnan kung saan ang pinakamalapit na evacuation center para sa iyo at sa iyong aso. '}

## V. POI search (`find_nearby`)

Rule 3: requests to find a facility map to `find_nearby` with the right `category` enum.

In [14]:
show("Hospital",          "Where's the closest hospital?", context="earthquake")


▶ Hospital
  user: "Where's the closest hospital?"
  model: {"action": "tool", "name": "find_nearby", "args": {"category": "hospital"}}



{'action': 'tool', 'name': 'find_nearby', 'args': {'category': 'hospital'}}

In [15]:
show("School (Tagalog)",  "May malapit bang paaralan dito?", context="typhoon")

▶ School (Tagalog)
  user: 'May malapit bang paaralan dito?'
  model: {"action": "speak", "text": "Maria, para makita ko kung may malapit na paaralan, kailangan ko ng kaunting impormasyon. Anong lugar ang tinutukoy mo? Kung gusto mong hanapin ang pinakamalapit na paaralan sa inyong lugar, pwede kong gamitin ang find_nearby tool."}



{'action': 'speak',
 'text': 'Maria, para makita ko kung may malapit na paaralan, kailangan ko ng kaunting impormasyon. Anong lugar ang tinutukoy mo? Kung gusto mong hanapin ang pinakamalapit na paaralan sa inyong lugar, pwede kong gamitin ang find_nearby tool.'}

In [16]:
show("Covered court",     "Is there a covered court nearby we can shelter in?", context="typhoon")

▶ Covered court
  user: 'Is there a covered court nearby we can shelter in?'
  model: {"action": "tool", "name": "find_nearby", "args": {"category": "covered_court"}}



{'action': 'tool',
 'name': 'find_nearby',
 'args': {'category': 'covered_court'}}

## VI. Profile-aware answers (`get_user_profile`)

Rule 4: self-referential questions call `get_user_profile`. Compare these against the `DEMO_PROFILE` defined in Section II.

In [17]:
show("Meds question",        "What medical conditions do I have on file?", context="earthquake")


▶ Meds question
  user: 'What medical conditions do I have on file?'
  model: {"action": "speak", "text": "Maria, your profile indicates you have asthma."}



{'action': 'speak', 'text': 'Maria, your profile indicates you have asthma.'}

In [18]:
show("Meeting point",        "Saan ulit yung meeting place namin ng pamilya?", context="typhoon")

▶ Meeting point
  user: 'Saan ulit yung meeting place namin ng pamilya?'
  model: {"action": "speak", "text": "Maria, ang primary meeting point ninyo ay nasa Rizal Park flagpole. Kung may emergency, doon kayo magpunta."}



{'action': 'speak',
 'text': 'Maria, ang primary meeting point ninyo ay nasa Rizal Park flagpole. Kung may emergency, doon kayo magpunta.'}

In [19]:
show("Emergency contacts",   "Who are my emergency contacts?", context="earthquake")

▶ Emergency contacts
  user: 'Who are my emergency contacts?'
  model: {"action": "speak", "text": "Hello Maria. Your emergency contacts are: Juan Dela Cruz (brother)."}



{'action': 'speak',
 'text': 'Hello Maria. Your emergency contacts are: Juan Dela Cruz (brother).'}

## VII. Multilingual (Rule 8)

The model replies in the **user's language** — English, Filipino, or Taglish. These all ask the same thing in different languages; watch the `speak` text (after the tool round-trip) match the input language.

In [20]:
show("English", "What should I do during an earthquake?", context="earthquake")

▶ English
  user: 'What should I do during an earthquake?'
  model: {"action": "speak", "text": "For earthquake safety, please follow the protocol. I need to check the official instructions now."}



{'action': 'speak',
 'text': 'For earthquake safety, please follow the protocol. I need to check the official instructions now.'}

In [21]:
show("Filipino", "Ano ang dapat kong gawin kapag may lindol?", context="earthquake")

▶ Filipino
  user: 'Ano ang dapat kong gawin kapag may lindol?'
  model: {"action": "speak", "text": "Maria, kapag may lindol, sundin ang mga sumusunod: Mag-Duck, Cover, and Hold. Magtago sa ilalim ng matibay na mesa o kama, at manatili roon hanggang sa humupa ang pagyanig. Huwag tumakbo palabas habang nagaganap ang lindol. Kung nasa loob ka ng bahay, iwasan ang mga bintana, salamin, at mabibigat na bagay. Para sa anumang safety protocol, kailangan kong tawagin ang get_protocol."}



{'action': 'speak',
 'text': 'Maria, kapag may lindol, sundin ang mga sumusunod: Mag-Duck, Cover, and Hold. Magtago sa ilalim ng matibay na mesa o kama, at manatili roon hanggang sa humupa ang pagyanig. Huwag tumakbo palabas habang nagaganap ang lindol. Kung nasa loob ka ng bahay, iwasan ang mga bintana, salamin, at mabibigat na bagay. Para sa anumang safety protocol, kailangan kong tawagin ang get_protocol.'}

In [22]:
show("Taglish",  "Pano ba mag-prepare for earthquake? Anong mga steps?", context="earthquake")

▶ Taglish
  user: 'Pano ba mag-prepare for earthquake? Anong mga steps?'
  model: {"action": "speak", "text": "Hello Maria. Para maghanda sa lindol, sundin ang mga sumusunod na hakbang: \n\n1. **Maghanda Bago ang Lindol (Before the Earthquake):**\n    * Maglagay ng mga gamit na madaling abutin sa isang ligtas na lugar. Huwag mag-panic.\n    * Alamin kung nasaan ang iyong emergency kit at tiyakin na ito ay kumpleto.\n    * Ihanda ang mga ruta papunta sa evacuation area.\n\n2. **Habang Nagaganap ang Lindol (During the Earthquake):**\n    * **Dapitan, Takpan, Pigilan (Drop, Cover, Hold On).** Agad humiga sa lupa, magtago sa ilalim ng matibay na mesa o kama, at kumapit nang mahigpit dito hanggang matapos ang pagyanig.\n    * Huwag tumakbo o magmadali habang nagaganap ang lindol.\n\n3. **Pagkatapos ng Lindol (After the Earthquake):**\n    * Mag-ingat sa mga sugat at anumang sira. Huwag gumamit ng kandila o kuryente kung may amoy ng gas.\n    * Sundin ang mga utos ng awtoridad para sa evacua

{'action': 'speak',
 'text': 'Hello Maria. Para maghanda sa lindol, sundin ang mga sumusunod na hakbang: \n\n1. **Maghanda Bago ang Lindol (Before the Earthquake):**\n    * Maglagay ng mga gamit na madaling abutin sa isang ligtas na lugar. Huwag mag-panic.\n    * Alamin kung nasaan ang iyong emergency kit at tiyakin na ito ay kumpleto.\n    * Ihanda ang mga ruta papunta sa evacuation area.\n\n2. **Habang Nagaganap ang Lindol (During the Earthquake):**\n    * **Dapitan, Takpan, Pigilan (Drop, Cover, Hold On).** Agad humiga sa lupa, magtago sa ilalim ng matibay na mesa o kama, at kumapit nang mahigpit dito hanggang matapos ang pagyanig.\n    * Huwag tumakbo o magmadali habang nagaganap ang lindol.\n\n3. **Pagkatapos ng Lindol (After the Earthquake):**\n    * Mag-ingat sa mga sugat at anumang sira. Huwag gumamit ng kandila o kuryente kung may amoy ng gas.\n    * Sundin ang mga utos ng awtoridad para sa evacuation. \n\nTandaan, Maria, dahil mayroon kang **asthma**, mas mahalaga na maging h

## VIII. Refusal of off-topic queries (Rule 7)

Out of scope by design — the model politely refuses and redirects to disaster topics instead of answering. It should emit a `speak` turn (no tool call).

In [23]:
show("Entertainment",     "Who won the NBA finals last year?", context="earthquake")

▶ Entertainment
  user: 'Who won the NBA finals last year?'
  model: {"action": "speak", "text": "I can only provide information related to disaster preparedness, safety, and official protocols for the Philippines. Please ask me about earthquake, typhoon, or volcano safety."}



{'action': 'speak',
 'text': 'I can only provide information related to disaster preparedness, safety, and official protocols for the Philippines. Please ask me about earthquake, typhoon, or volcano safety.'}

In [24]:
show("General knowledge", "Write me a poem about the moon.", context="typhoon")

▶ General knowledge
  user: 'Write me a poem about the moon.'
  model: {"action": "speak", "text": "I can only provide information and assistance related to disasters, safety, and emergency protocols. Please ask me about typhoons, earthquakes, or other disaster-related topics."}



{'action': 'speak',
 'text': 'I can only provide information and assistance related to disasters, safety, and emergency protocols. Please ask me about typhoons, earthquakes, or other disaster-related topics.'}

In [25]:
show("Opinion",           "What's your favorite food?", context="volcano")

▶ Opinion
  user: "What's your favorite food?"
  model: {"action": "speak", "text": "As LIKAS, I don't have personal preferences like favorite foods. My focus is entirely on providing you with the most accurate and life-saving information during a disaster. Are you asking about safety protocols or evacuation routes for Maria?"}



{'action': 'speak',
 'text': "As LIKAS, I don't have personal preferences like favorite foods. My focus is entirely on providing you with the most accurate and life-saving information during a disaster. Are you asking about safety protocols or evacuation routes for Maria?"}

## IX. The full multi-turn tool-dispatch loop

In production the host runs the tool, injects the result, and the model decides again — capped at 3 iterations (see `MAX_TOOL_CALLS_PER_TURN` in `aiAssistantService.ts`). Here we **mock** the tool results with strings shaped exactly like `aiTools.ts` returns, so you can see a complete `tool → result → speak` exchange.

This is the realistic end-to-end behavior — a single bare turn (Sections III–VIII) usually stops at the tool call; the verbatim-quoting and personalization happen on the **second** turn after the result comes back.

In [36]:
# Mocked tool results, 1:1 in shape with Likas/src/services/aiTools.ts.
MOCK_TOOL_RESULTS = {
    "get_protocol": (
        "PHIVOLCS and NDRRMC guidance: DROP, COVER, AND HOLD ON. Get under a "
        "sturdy table and hold its leg. Stay away from windows and heavy "
        "furniture. After shaking stops, check for injuries, avoid elevators, "
        "and watch for aftershocks."
    ),
    "route_to_nearest_evacuation": (
        "Top evacuation options:\n"
        "1. Manila High School — 0.8 km (~11 min walking) [best match]\n"
        "2. Rajah Soliman Park Covered Court — 1.3 km (~18 min walking)\n"
        "3. Baseco Sports Complex — 2.1 km (~29 min walking)\n\n"
        "Route to Manila High School: 0.83 km along walkable roads, "
        "~11 min walking."
    ),
    "find_nearby": (
        "Nearest hospital:\n"
        "1. Ospital ng Maynila — 1.2 km · Quirino Ave cor Roxas Blvd\n"
        "2. Manila Doctors Hospital — 1.9 km · 667 United Nations Ave\n"
        "3. Philippine General Hospital — 2.4 km · Taft Ave, Ermita"
    ),
    "get_user_profile": (
        "Name: Maria\nAge group: adult\n"
        "Companions: 1 infants, 0 children, 1 elderly, 0 PWD\n"
        "Pets: 1 dogs (medium)\nMedical conditions: asthma\n"
        "Address: 12 Mabini St, Barangay 659, Manila\n"
        "Primary meeting point: Rizal Park flagpole\n"
        "Emergency contacts: Juan Dela Cruz — brother: 0917xxxxxxx"
    ),
}


def dispatch_loop(user_message, context="earthquake", profile=DEMO_PROFILE,
                  max_turns=3):
    """Full loop: model -> (tool -> mocked result -> model)* -> speak.

    NOTE: Gemma's chat template only accepts the `user` and `assistant` roles
    and enforces strict user/assistant alternation. There is no `tool` role.
    So the tool result is fed back as a `user`-role message (this is also how
    the on-device app injects it), and follow-up turns call dispatch with
    user_message=None so we don't append a second consecutive user turn.
    """
    print("=" * 72)
    print(f"▶ {user_message!r}  [context={context}]")
    history = []
    for turn in range(max_turns):
        # turn 0: pass the real question. Later turns: None — the latest
        # history entry is already the tool-result user message.
        result = dispatch(user_message if turn == 0 else None, context,
                          profile, history=history or None)
        if turn == 0:
            history.append({"role": "user", "content": user_message})
        print(f"  [turn {turn}] model:", json.dumps(result, ensure_ascii=False))
        if result.get("action") == "speak":
            print(f"\n  ✓ FINAL REPLY:\n  {result['text']}\n")
            return result
        if result.get("action") == "tool":
            name = result["name"]
            tool_result = MOCK_TOOL_RESULTS.get(name, "(no mock result)")
            print(f"  [turn {turn}] tool({name}) -> {tool_result[:70]}...")
            # assistant = the tool call it just made; then the result comes
            # back as a USER turn (Gemma has no `tool` role). This keeps the
            # user/assistant alternation valid.
            history.append({"role": "assistant", "content": json.dumps(result)})
            history.append({
                "role": "user",
                "content": (
                    f"Tool result for {name}:\n{tool_result}\n\n"
                    "Now respond to my original request using this result."
                ),
            })
            continue
        print("  (unparsed / unexpected envelope)")
        return result
    print("  (hit tool-call limit without a speak turn)")
    return None

In [27]:
# Watch the model quote the protocol VERBATIM on the second turn (Rule 1)
# and personalize for Maria's asthma / elderly companion / dog (Rule 5).
dispatch_loop("May lindol! Ano gagawin ko?", context="earthquake")

▶ 'May lindol! Ano gagawin ko?'  [context=earthquake]
  [turn 0] model: {"action": "speak", "text": "Maria, kung may lindol, sundin agad ang mga hakbang na ito. Una, maghanap ng ligtas na lugar at magtago sa ilalim ng matibay na mesa o kama. Huwag tumakbo sa gitna ng pagyanig. Pagkatapos humupa ang lindol, maging handa para sa posibleng aftershocks. Kung ikaw ay may hininga (asthma), magsuot agad ng mask. Para sa evacuation, gamitin mo ang route_to_nearest_evacuation."}

  ✓ FINAL REPLY:
  Maria, kung may lindol, sundin agad ang mga hakbang na ito. Una, maghanap ng ligtas na lugar at magtago sa ilalim ng matibay na mesa o kama. Huwag tumakbo sa gitna ng pagyanig. Pagkatapos humupa ang lindol, maging handa para sa posibleng aftershocks. Kung ikaw ay may hininga (asthma), magsuot agad ng mask. Para sa evacuation, gamitin mo ang route_to_nearest_evacuation.



{'action': 'speak',
 'text': 'Maria, kung may lindol, sundin agad ang mga hakbang na ito. Una, maghanap ng ligtas na lugar at magtago sa ilalim ng matibay na mesa o kama. Huwag tumakbo sa gitna ng pagyanig. Pagkatapos humupa ang lindol, maging handa para sa posibleng aftershocks. Kung ikaw ay may hininga (asthma), magsuot agad ng mask. Para sa evacuation, gamitin mo ang route_to_nearest_evacuation.'}

In [37]:
dispatch_loop("Saan kami pwedeng lumikas? May aso at lola ako.", context="typhoon")

▶ 'Saan kami pwedeng lumikas? May aso at lola ako.'  [context=typhoon]
  [turn 0] model: {"action": "tool", "name": "route_to_nearest_evacuation", "args": {}}
  [turn 0] tool(route_to_nearest_evacuation) -> Top evacuation options:
1. Manila High School — 0.8 km (~11 min walkin...
  [turn 1] model: {"action": "speak", "text": "Maria, dahil may aso at lola ka, inirerekomenda ko ang Manila High School bilang pinakamalapit na evacuation center. Ang ruta papunta doon ay 0.83 km sa mga walkable roads, tinatayang 11 minuto ang lakad. Siguraduhin ninyong isama ang aso at lola sa inyong paglikas."}

  ✓ FINAL REPLY:
  Maria, dahil may aso at lola ka, inirerekomenda ko ang Manila High School bilang pinakamalapit na evacuation center. Ang ruta papunta doon ay 0.83 km sa mga walkable roads, tinatayang 11 minuto ang lakad. Siguraduhin ninyong isama ang aso at lola sa inyong paglikas.



{'action': 'speak',
 'text': 'Maria, dahil may aso at lola ka, inirerekomenda ko ang Manila High School bilang pinakamalapit na evacuation center. Ang ruta papunta doon ay 0.83 km sa mga walkable roads, tinatayang 11 minuto ang lakad. Siguraduhin ninyong isama ang aso at lola sa inyong paglikas.'}

In [29]:
dispatch_loop("What meds do I have, and what should I pack?", context="earthquake")

▶ 'What meds do I have, and what should I pack?'  [context=earthquake]
  [turn 0] model: {"action": "speak", "text": "Hello Maria. I can check your profile for you, but for critical safety information, please follow the evacuation protocols. To give you the best advice on what to pack, I need to know your profile first."}

  ✓ FINAL REPLY:
  Hello Maria. I can check your profile for you, but for critical safety information, please follow the evacuation protocols. To give you the best advice on what to pack, I need to know your profile first.



{'action': 'speak',
 'text': 'Hello Maria. I can check your profile for you, but for critical safety information, please follow the evacuation protocols. To give you the best advice on what to pack, I need to know your profile first.'}

## X. Try your own

Edit `DEMO_PROFILE` in Section II (re-run that cell) to change personalization, then call `dispatch_loop(...)` with any prompt. Things worth probing:

- **Multi-tool**: "Lindol! Saan ako pupunta?" — protocol *then* evacuation in one conversation.
- **Clarification**: "help" / "it's bad" — the model asks what's wrong before acting.
- **Language stickiness**: ask in Taglish and confirm the final `speak.text` stays Taglish.
- **Refusal**: any off-topic query should be a single `speak` turn that redirects.

See the [model card](https://huggingface.co/jpcurada/likas-gemma4-e2b-gguf) for the full evaluation rubric and limitations (Metro Manila scope, authority-snapshot drift, JSON-envelope requirement).

In [39]:
dispatch_loop("Hello po, ano kaya mong gawin?")

▶ 'Hello po, ano kaya mong gawin?'  [context=earthquake]
  [turn 0] model: {"action": "speak", "text": "Hello Maria. Since you are asking what to do, I need more information about the situation. Are you currently experiencing an emergency? Please tell me what disaster you are referring to."}

  ✓ FINAL REPLY:
  Hello Maria. Since you are asking what to do, I need more information about the situation. Are you currently experiencing an emergency? Please tell me what disaster you are referring to.



{'action': 'speak',
 'text': 'Hello Maria. Since you are asking what to do, I need more information about the situation. Are you currently experiencing an emergency? Please tell me what disaster you are referring to.'}